<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# 5G Network Operator Agent with NVIDIA NIM

This notebook demonstrates an educational decision loop over deterministic synthetic observations. It does not train a model, connect to a live RAN or other physical infrastructure, or claim physical-network fidelity.

## Outcome and prerequisites

Follow the [README](README.md) to create the Python environment. The local noop and scripted-relief baselines always run; the hosted NVIDIA NIM policy is optional and requires an existing `NVIDIA_API_KEY`.

In [ ]:
import json
import os
from math import isfinite
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from network_environment import (
    NetworkEnvironment,
    default_scenario,
    noop_policy,
    render_observation,
    run_episode,
    scripted_relief_policy,
    tool_schemas,
)

load_dotenv()

def ue_frame(observation):
    return pd.DataFrame([ue.to_dict() for ue in observation.ues]).set_index("ue_id")

def tool_frame(tools):
    return pd.DataFrame(
        {
            "action": tool["function"]["name"],
            "description": tool["function"]["description"],
        }
        for tool in tools
    )

def result_frame(episodes):
    return pd.DataFrame(
        {"policy": name, "turns": len(episode.transitions), "total_score": episode.total_reward}
        for name, episode in episodes.items()
    ).set_index("policy")

def plot_cumulative_scores(episodes):
    figure, axis = plt.subplots(figsize=(7, 4))
    for name, episode in episodes.items():
        cumulative = 0.0
        values = [0.0]
        for transition in episode.transitions:
            cumulative += transition.reward["total"]
            values.append(cumulative)
        axis.plot(range(len(values)), values, marker="o", label=name)
    axis.set(xlabel="turn", ylabel="cumulative score", title="Same synthetic scenario and four-turn horizon")
    axis.legend()
    axis.grid(alpha=0.25)
    plt.show()


In [ ]:
environment = NetworkEnvironment(default_scenario())
initial_observation = environment.reset()
print(render_observation(initial_observation))
display(ue_frame(initial_observation))
display(pd.DataFrame([initial_observation.cell.to_dict()], index=["cell"]))
display(tool_frame(tool_schemas()))


## Transparent scoring

Every score term is non-positive and zero is ideal. The total is the exact sum of unmet-throughput, unfairness, SLA-violation, PRB-pressure, and rejected-action costs. Returns are comparable only on the same bundled scenario and horizon; these baseline values are neither model results nor network-performance measurements.

In [ ]:
MAX_STEPS = 4
scenario = default_scenario()
episodes = {
    "noop": run_episode(noop_policy, scenario=scenario, max_steps=MAX_STEPS),
    "scripted_relief": run_episode(scripted_relief_policy, scenario=scenario, max_steps=MAX_STEPS),
}
display(result_frame(episodes))
plot_cumulative_scores(episodes)


### Reading the baseline comparison

Both policies receive the same initial state and exactly four decisions. Scripted relief is a fixed local rule for this scenario, not a learned policy.

## Optional hosted NVIDIA NIM policy

Set `NVIDIA_API_KEY` before launching the notebook to enable one OpenAI-compatible hosted policy. `NVIDIA_MODEL` can override the default model. Without a key, only hosted-policy evaluation is skipped; every offline table, assertion, and chart still runs.

In [ ]:
api_key = os.getenv("NVIDIA_API_KEY")
model_id = os.getenv("NVIDIA_MODEL", "nvidia/nemotron-3-super-120b-a12b")
client = None
if not api_key:
    print("NVIDIA_API_KEY is not set; hosted-policy evaluation is skipped.")
else:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=api_key)


## Strict one-call contract

One request must yield exactly one function call. A missing, multiple, non-function, unparseable, non-object, or request-error response becomes a bounded `__invalid_model_output__` action. The environment records that action as rejected with a finite penalty; no failure is silently replaced by `noop`. Unknown tools and invalid arguments pass through unchanged so the same action validator records their exact rejection.

In [ ]:
def _invalid_model_action(reason):
    return {"name": f"__invalid_model_output__:{reason}", "arguments": {}}

def _parse_tool_call_response(response):
    try:
        choices = list(response.choices)
    except (AttributeError, TypeError):
        return _invalid_model_action("unexpected_choice_count")
    if len(choices) != 1:
        return _invalid_model_action("unexpected_choice_count")

    try:
        tool_calls = list(choices[0].message.tool_calls or [])
    except (AttributeError, TypeError):
        return _invalid_model_action("missing_tool_call")
    if not tool_calls:
        return _invalid_model_action("missing_tool_call")
    if len(tool_calls) != 1:
        return _invalid_model_action("multiple_tool_calls")

    call = tool_calls[0]
    function = getattr(call, "function", None)
    name = getattr(function, "name", None)
    arguments = getattr(function, "arguments", None)
    if getattr(call, "type", None) != "function" or not isinstance(name, str) or not name:
        return _invalid_model_action("non_function_call")
    if not isinstance(arguments, str):
        return _invalid_model_action("invalid_json_arguments")
    try:
        decoded_arguments = json.loads(arguments)
    except (json.JSONDecodeError, TypeError):
        return _invalid_model_action("invalid_json_arguments")
    if not isinstance(decoded_arguments, dict):
        return _invalid_model_action("non_object_arguments")
    return {"name": name, "arguments": decoded_arguments}

def _request_error_action(error):
    error_class = "".join(
        character.lower() if character.isalnum() else "_"
        for character in type(error).__name__
    )[:48].strip("_") or "exception"
    return _invalid_model_action(f"request_error_{error_class}")

def nim_policy(observation, tools, *, _client=None):
    active_client = _client if _client is not None else client
    messages = [
        {
            "role": "system",
            "content": (
                "Select exactly one provided tool for the deterministic synthetic 5G observation. "
                "Do not invent tools or return prose instead of a tool call."
            ),
        },
        {"role": "user", "content": json.dumps(observation.to_dict(), sort_keys=True)},
    ]
    try:
        response = active_client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=tools,
            tool_choice="required",
            parallel_tool_calls=False,
            n=1,
            temperature=1.0,
            top_p=0.95,
            max_tokens=1024,
        )
    except Exception as error:
        return _request_error_action(error)
    return _parse_tool_call_response(response)

def _fake_call(name, arguments, call_type="function"):
    return SimpleNamespace(
        type=call_type,
        function=SimpleNamespace(name=name, arguments=arguments),
    )

def _fake_response(tool_calls, choice_count=1):
    return SimpleNamespace(
        choices=[SimpleNamespace(message=SimpleNamespace(tool_calls=tool_calls)) for _ in range(choice_count)]
    )

valid = _parse_tool_call_response(_fake_response([_fake_call("noop", "{}")]))
assert valid == {"name": "noop", "arguments": {}}

invalid_fixtures = {
    "missing_tool_call": _fake_response([]),
    "multiple_tool_calls": _fake_response([_fake_call("noop", "{}"), _fake_call("noop", "{}")]),
    "non_function_call": _fake_response([_fake_call("noop", "{}", call_type="custom")]),
    "invalid_json_arguments": _fake_response([_fake_call("noop", "{")]),
    "non_object_arguments": _fake_response([_fake_call("noop", "[]")]),
    "unexpected_choice_count": _fake_response([], choice_count=0),
}
assert _parse_tool_call_response(_fake_response([], choice_count=2)) == {
    "name": "__invalid_model_output__:unexpected_choice_count",
    "arguments": {},
}
for reason, response in invalid_fixtures.items():
    action = _parse_tool_call_response(response)
    assert action == {"name": f"__invalid_model_output__:{reason}", "arguments": {}}
    fixture_environment = NetworkEnvironment(default_scenario())
    fixture_environment.reset()
    rejected = fixture_environment.step(action)
    assert not rejected.accepted and isfinite(rejected.reward["rejected_action"])
    assert rejected.action["name"] != "noop"

unknown_action = _parse_tool_call_response(_fake_response([_fake_call("unknown_tool", "{}")]))
assert unknown_action == {"name": "unknown_tool", "arguments": {}}
unknown_environment = NetworkEnvironment(default_scenario())
unknown_environment.reset()
assert not unknown_environment.step(unknown_action).accepted

class _FailingCompletions:
    def create(self, **_kwargs):
        raise TimeoutError

failing_client = SimpleNamespace(chat=SimpleNamespace(completions=_FailingCompletions()))
request_error_action = nim_policy(initial_observation, tool_schemas(), _client=failing_client)
assert request_error_action == {"name": "__invalid_model_output__:request_error_timeouterror", "arguments": {}}
request_error_environment = NetworkEnvironment(default_scenario())
request_error_environment.reset()
request_error_transition = request_error_environment.step(request_error_action)
assert not request_error_transition.accepted
assert request_error_transition.action["name"] != "noop"
assert isfinite(request_error_transition.reward["rejected_action"])
print("Offline parser assertions passed: valid call and seven invalid-output classes.")


## Episode comparison and transition inspection

If configured, the hosted policy runs on the same four-turn scenario and is added to the table and chart. The detailed view chooses its first rejected transition when available; otherwise it shows the first scripted-relief transition. JSON export is intentionally opt-in.

In [ ]:
hosted_episode = None
if client is not None:
    hosted_episode = run_episode(nim_policy, scenario=scenario, max_steps=MAX_STEPS)
    episodes["hosted_nim"] = hosted_episode
    display(result_frame(episodes))
    plot_cumulative_scores(episodes)
    display(pd.DataFrame(
        {"accepted": transition.accepted, "error": transition.error}
        for transition in hosted_episode.transitions
    ))

rejected_hosted = (
    next((transition for transition in hosted_episode.transitions if not transition.accepted), None)
    if hosted_episode is not None
    else None
)
selected_transition = rejected_hosted or episodes["scripted_relief"].transitions[0]
print("Before")
display(pd.DataFrame([selected_transition.before.cell.to_dict()], index=["cell"]))
display(ue_frame(selected_transition.before))
print("Action / validation")
display(pd.DataFrame([{
    "action": json.dumps(selected_transition.action, sort_keys=True),
    "accepted": selected_transition.accepted,
    "error": selected_transition.error,
}]))
print("After")
display(pd.DataFrame([selected_transition.after.cell.to_dict()], index=["cell"]))
display(ue_frame(selected_transition.after))
display(pd.DataFrame(
    {"score_term": name, "value": value}
    for name, value in selected_transition.reward.items()
))

EXPORT_ROLLOUT = False
if EXPORT_ROLLOUT:
    selected_episode = hosted_episode or episodes["scripted_relief"]
    output_path = Path("outputs") / "5g_network_operator_rollout.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(selected_episode.to_dict(), indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    print(f"Wrote {output_path.as_posix()}")
else:
    print("EXPORT_ROLLOUT is False; no JSON file was written.")


## Limitations and next steps

This example is deterministic, synthetic, single-cell, and single-scenario. It performs no training, recorded playback, database access, live-network actuation, or physical-fidelity validation. Treat its scores only as transparent educational costs within this fixed model. Next, try another bounded action or opt into a hosted policy, then inspect the transition rather than inferring broader network performance. See the [NVIDIA API documentation](https://docs.api.nvidia.com/) and [`network_environment.py`](network_environment.py).